# Tutorial 02 -- Units, Frames, and Conventions

Check the conventions that matter most for cQED users: SI-style internal units, rotating-frame interpretation, positive drive-frequency translation, and the meaning of the dispersive `chi` sign.

**Prerequisites.** Tutorial 01 is recommended first.


## 1. Goal

This tutorial turns the most important convention questions into explicit notebook checks so you can verify them before running large parameter sweeps.


## 2. Physical Background

Three convention choices matter constantly in `cqed_sim`:

1. internal angular frequencies are in `rad/s`, not in `Hz`
2. rotating-frame frequencies are specified through `FrameSpec`
3. user-facing code should work with positive physical drive tones, while the low-level `Pulse.carrier` compatibility field still uses the waveform convention `exp(+i (omega t + phase))`, so the resonant raw carrier is the negative of the rotating-frame transition frequency

A fourth convention is the repository-wide interpretation of runtime `chi`: negative `chi` lowers the qubit transition as bosonic occupation increases.


## 3. Imports


In [ ]:

from pathlib import Path
import sys

REPO_ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "pyproject.toml").exists() and (candidate / "cqed_sim").is_dir()
    ),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Could not resolve the repository root from the notebook working directory.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import matplotlib.pyplot as plt
import numpy as np

from cqed_sim import (
    DispersiveTransmonCavityModel,
    FrameSpec,
    Pulse,
    compute_energy_spectrum,
    drive_frequency_for_transition_frequency,
    internal_carrier_from_drive_frequency,
    manifold_transition_frequency,
    transition_frequency_from_drive_frequency,
)
from cqed_sim.plotting import plot_energy_levels
from tutorials.tutorial_support import (
    GHz,
    MHz,
    angular_to_mhz,
)

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (7.0, 4.2)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

## 4. Simulation Parameters


In [ ]:
chi_values_hz = (-2.5e6, +2.5e6)
n_values = np.arange(6)
transition_detuning_demo_mhz = np.array([-3.0, -1.5, 0.0, 1.5, 3.0])


## 5. Model Construction


In [ ]:
models = {
    f"chi = {chi_hz / 1.0e6:+.1f} MHz": DispersiveTransmonCavityModel(
        omega_c=GHz(5.0),
        omega_q=GHz(6.2),
        alpha=MHz(-220.0),
        chi=MHz(chi_hz / 1.0e6),
        kerr=0.0,
        n_cav=8,
        n_tr=2,
    )
    for chi_hz in chi_values_hz
}
frame = FrameSpec(omega_c_frame=GHz(5.0), omega_q_frame=GHz(6.2))

transition_demo = MHz(transition_detuning_demo_mhz[1])
drive_frequency_demo = drive_frequency_for_transition_frequency(transition_demo, frame.omega_q_frame)
carrier_demo = internal_carrier_from_drive_frequency(drive_frequency_demo, frame.omega_q_frame)
round_trip_demo = angular_to_mhz(
    transition_frequency_from_drive_frequency(drive_frequency_demo, frame.omega_q_frame)
)
print(f"Example transition detuning = {transition_detuning_demo_mhz[1]:+.3f} MHz")
print(f"drive_frequency_for_transition_frequency(...) returns {angular_to_mhz(drive_frequency_demo):+.3f} MHz as a positive physical tone")
print(f"internal_carrier_from_drive_frequency(...) returns {angular_to_mhz(carrier_demo):+.3f} MHz as the raw carrier")
print(f"transition_frequency_from_drive_frequency(...) returns the original transition detuning: {round_trip_demo:+.3f} MHz")


## 6. Pulse / Sequence Construction


In [ ]:
transition_curves = {
    label: [angular_to_mhz(manifold_transition_frequency(model, int(n), frame=frame)) for n in n_values]
    for label, model in models.items()
}


## 7. Running the Simulation


In [ ]:
reference_model = models["chi = -2.5 MHz"]
lab_spectrum = compute_energy_spectrum(reference_model, frame=FrameSpec(), levels=8)
rot_spectrum = compute_energy_spectrum(reference_model, frame=frame, levels=8)
print("Vacuum-referenced lab-frame energies [MHz]:")
print(np.round(lab_spectrum.energies[:6] / (2.0 * np.pi * 1.0e6), 4))
print("Vacuum-referenced rotating-frame energies [MHz]:")
print(np.round(rot_spectrum.energies[:6] / (2.0 * np.pi * 1.0e6), 4))


## 8. Visualizing the Results


In [ ]:
fig, ax = plt.subplots()
for label, values in transition_curves.items():
    ax.plot(n_values, values, "o-", label=label)
ax.axhline(0.0, color="black", linewidth=1.0, alpha=0.5)
ax.set_xlabel("Bosonic occupation n")
ax.set_ylabel(r"$\omega_{ge}(n) / 2\pi$ in the chosen rotating frame [MHz]")
ax.set_title("The sign of runtime chi controls the slope of the qubit line")
ax.legend()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
plot_energy_levels(lab_spectrum, max_levels=8, energy_scale=1.0 / (2.0 * np.pi * 1.0e6), energy_unit_label="MHz", title="Lab frame", ax=axes[0])
plot_energy_levels(rot_spectrum, max_levels=8, energy_scale=1.0 / (2.0 * np.pi * 1.0e6), energy_unit_label="MHz", title="Matched rotating frame", ax=axes[1])
plt.show()


## 9. Physical Interpretation

A spectroscopy axis should be labeled by physical transition detuning, not by the raw carrier. The preferred user-facing path is to map that detuning into a positive physical drive tone with `drive_frequency_for_transition_frequency(...)` and only translate to the low-level raw carrier with `internal_carrier_from_drive_frequency(...)` at the pulse boundary. The frame comparison also shows why rotating-frame energies can cluster near zero even when the lab-frame spectrum still sits near several gigahertz.


## 10. Exercises / Next Steps

- Change the rotating frame to be intentionally off-resonant and see how every dressed level shifts.
- Open `tutorials/conventions_quick_reference.md` beside this notebook and confirm that each rule is reflected in the code above.
- Continue to Tutorial 06 before doing any spectroscopy work.
